In [ ]:
import pandas as pd
import re
from pathlib import Path


INPUT_FOLDER = "./"
OUTPUT_FOLDER = "grouped_keywords"


def clean_keyword_string(text):
    """
    Remove special characters EXCEPT commas.
    """
    if pd.isna(text):
        return ""

    text = str(text)

    # Keep letters, numbers, whitespace and commas
    text = re.sub(r"[^\w\s,]", "", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


input_path = Path(INPUT_FOLDER)
output_path = Path(OUTPUT_FOLDER)
output_path.mkdir(exist_ok=True)

groups = {
    "age": {"keywords": set(), "files": []},
    "nationality": {"keywords": set(), "files": []},
    "gender": {"keywords": set(), "files": []},
    "disorder": {"keywords": set(), "files": []}
}

for csv_file in input_path.glob("*.csv"):

    filename = csv_file.name.lower()

    group = None

    if "age" in filename:
        group = "age"
    elif "nat" in filename:
        group = "nationality"
    elif "gender" in filename:
        group = "gender"
    elif "disorder" in filename:
        group = "disorder"



    if group is None:
        continue

    groups[group]["files"].append(csv_file.name)

    try:
        df = pd.read_csv(csv_file)

        if "keyword" not in df.columns:
            print(f"Skipping {csv_file.name}: no keyword column")
            continue

        for value in df["keyword"]:

            #cleaned = clean_keyword_string(value)
            cleaned = value.strip().lower()

            # Split on commas and treat each item separately
            keywords = cleaned.split(",")

            for kw in keywords:

                kw = kw.strip()

                if kw:
                    groups[group]["keywords"].add(kw)

    except Exception as e:
        print(f"Error reading {csv_file.name}: {e}")


# Name outputfiles as you wish, but make sure they are unique
output_files = {
    "age": "age_keywords.csv",
    "nationality": "nationality_keywords.csv",
    "gender": "gender_keywords.csv",
    "disorder": "disorder_keywords.csv"
}

for group_name, info in groups.items():

    output_file = output_path / output_files[group_name]

    pd.DataFrame(
        sorted(info["keywords"]),
        columns=["keyword"]
    ).to_csv(output_file, index=False)


print("\n" + "=" * 60)
print("GROUPING REPORT")
print("=" * 60)

for group_name, info in groups.items():

    print(f"\n{group_name.upper()}")

    for file in info["files"]:
        print(f"  - {file}")

    print(f"  Unique keywords: {len(info['keywords'])}")

In [ ]:
import pandas as pd
import re

# =====================================================
# INPUT FILES
# =====================================================

TERMS_FILE = "gender.csv"      # file with terms, do this for each file in Regex_Filters folder
POSTS_FILE = "user_content.csv"      # file with all posts (title/body) from a user. These files are not provided here due to anonymization and privacy concerns.
OUTPUT_FILE = "matched_posts.csv"
# =====================================================
# LOAD TERMS
# =====================================================

terms_df = pd.read_csv(TERMS_FILE)

# Assumes the terms file has a column named "keyword"
terms = (
    terms_df["keyword"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

print(f"Loaded {len(terms)} terms")

# =====================================================
# LOAD POSTS
# =====================================================

posts_df = pd.read_csv(POSTS_FILE)

required_columns = ["title", "body"]

for col in required_columns:
    if col not in posts_df.columns:
        raise ValueError(f"Column '{col}' not found in posts file")

# =====================================================
# SEARCH POSTS
# =====================================================

results = []

for _, row in posts_df.iterrows():

    title = str(row["title"]) if pd.notna(row["title"]) else ""
    body = str(row["body"]) if pd.notna(row["body"]) else ""

    full_text = f"{title} {body}"

    full_text_lower = full_text.lower()

    matched_terms = []

    for term in terms:

        term_lower = term.lower()

        # word-boundary match
        pattern = rf"\b{re.escape(term_lower)}\b"

        if re.search(pattern, full_text_lower):
            matched_terms.append(term)

    if matched_terms:

        output_row = row.to_dict()

        output_row["combined_text"] = full_text

        output_row["matched_terms"] = ", ".join(sorted(set(matched_terms)))

        output_row["n_terms_found"] = len(set(matched_terms))

        results.append(output_row)

# =====================================================
# SAVE RESULTS
# =====================================================

results_df = pd.DataFrame(results)

results_df.to_csv(OUTPUT_FILE, index=False)

print(f"\nPosts containing at least one term: {len(results_df)}")
print(f"Output saved to: {OUTPUT_FILE}")

In [ ]:
# Another example, now for age.csv
# Age received a different treatment because it was based only on the original_post column, not the set of all posts and comments from each user.

import pandas as pd
import re

# =====================================================
# INPUT FILES
# =====================================================

TERMS_FILE = "age.csv"
POSTS_FILE = "user_content.csv"
OUTPUT_FILE = "matched_posts_age.csv"

# =====================================================
# LOAD TERMS
# =====================================================

terms_df = pd.read_csv(TERMS_FILE)

# Assumes the terms file has a column named "keyword"
terms = (
    terms_df["keyword"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

print(f"Loaded {len(terms)} terms")

# =====================================================
# LOAD POSTS
# =====================================================

posts_df = pd.read_csv(POSTS_FILE)

if "original_post" not in posts_df.columns:
    raise ValueError("Column 'original_post' not found in posts file")

# =====================================================
# SEARCH POSTS
# =====================================================

results = []

for _, row in posts_df.iterrows():

    text = str(row["original_post"]) if pd.notna(row["original_post"]) else ""
    text_lower = text.lower()

    matched_terms = []

    for term in terms:

        pattern = rf"{re.escape(term.lower())}"

        if re.search(pattern, text_lower):
            matched_terms.append(term)

    if matched_terms:

        output_row = row.to_dict()

        output_row["matched_terms"] = ", ".join(sorted(set(matched_terms)))
        output_row["n_terms_found"] = len(set(matched_terms))

        results.append(output_row)

# =====================================================
# SAVE RESULTS
# =====================================================

results_df = pd.DataFrame(results)

results_df.to_csv(OUTPUT_FILE, index=False)

print(f"\nPosts containing at least one age-related term: {len(results_df)}")
print(f"Output saved to: {OUTPUT_FILE}")